In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")

In [ ]:
from ase.io import read
print("ASE imported")

In [ ]:
import torch
import numpy as np
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read, write
from ase.io.lammpsdata import read_lammps_data
from ase.optimize import BFGS, LBFGS
from ase.constraints import FixAtoms
from ase.filters import ExpCellFilter  # ← Changed from ase.constraints
from ase.build import bulk

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Imports complete")

In [ ]:
import shutil
from pathlib import Path

cache_dir = Path.home() / '.cache' / 'mace'
print(f"Cache directory: {cache_dir}")

if cache_dir.exists():
    print(f"\nContents of cache:")
    for file in cache_dir.iterdir():
        size_mb = file.stat().st_size / (1024**2)
        print(f"  {file.name}: {size_mb:.2f} MB")
    
    print(f"\nDeleting corrupted cache...")
    shutil.rmtree(cache_dir)
    print("✓ Cache deleted")
else:
    print("No cache found")

# Recreate empty cache directory
cache_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Fresh cache directory created: {cache_dir}")


In [ ]:
from mace.calculators import mace_mp
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

model_path = Path('mace-mh-1.model')

if model_path.exists():
    size_mb = model_path.stat().st_size / (1024**2)
    print(f"\n✓ Model found: {model_path}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f" Model not found at: {model_path.absolute()}")
    raise FileNotFoundError(f"Model file not found: {model_path}")

print(f"\nLoading MACE-MH-1 with OMAT/PBE head...")
print("(OMAT head trained on OC20 surface dataset - best for our application)")

mace_calculator = mace_mp(
    model=str(model_path),
    default_dtype="float64",
    device=device,
    head="omat_pbe" 
)

print("\n✓ MACE-MH-1 loaded successfully!")
print(f"  Model: {model_path.absolute()}")
print(f"  Head: omat_pbe (OC20 surface training)")
print(f"  Device: {device}")
print(f"  Precision: float64")


In [ ]:
from ase import Atoms

# Test with a simple molecule
test = Atoms('H2O', positions=[[0, 0, 0], [0, 0, 1], [0, 1, 0]])
test.calc = mace_calculator

energy = test.get_potential_energy()
forces = test.get_forces()

print("✓ MACE test successful!")
print(f"  Energy: {energy:.6f} eV")
print(f"  Max force: {np.max(np.linalg.norm(forces, axis=1)):.6f} eV/Å")
print("\n MACE-MH-1 (OMAT/PBE head) ready for validation!")

In [ ]:
# File paths to all clean slab structures
base_dirs = {
    'MgZn2': 'mgzn2_adsorption/slabs',
    'Mg4Zn7': 'mg4zn7_adsorption/slabs',
    'Mg21Zn25': 'mg21zn25_adsorption/slabs',
    'Mg2Zn11': 'mg2zn11_adsorption/slabs'
}

facets = {
    'MgZn2': ['001', '100', '101', '110'],
    'Mg4Zn7': ['001', '010', '100', '101', '110'],
    'Mg21Zn25': ['001', '100', '101', '110', '201'],
    'Mg2Zn11': ['100', '110', '111', '210', '211']
}

# Build full file paths
slab_files = {}
for compound, base_dir in base_dirs.items():
    slab_files[compound] = {}
    for facet in facets[compound]:
        if compound == 'MgZn2':
            filename = f'mgzn2_{facet}_reconstructed_ase.data'
        elif compound == 'Mg4Zn7':
            filename = f'mg4zn7_{facet}_reconstructed_ase.data'
        elif compound == 'Mg21Zn25':
            filename = f'mg21zn25_{facet}_reconstructed_ase.data'
        elif compound == 'Mg2Zn11':
            filename = f'mg2zn11_{facet}_6layers_reconstructed_ase.data'
        
        filepath = os.path.join(base_dir, filename)
        slab_files[compound][facet] = filepath

# Verify files exist
print("Verifying slab files...")
print("="*70)
missing_files = []
total_found = 0

for compound in slab_files:
    print(f"\n{compound}:")
    for facet, filepath in slab_files[compound].items():
        if os.path.exists(filepath):
            print(f"  ✓ ({facet}): {filepath}")
            total_found += 1
        else:
            print(f"  ✗ ({facet}): MISSING - {filepath}")
            missing_files.append(filepath)

print("\n" + "="*70)
if missing_files:
    print(f"Warning: {len(missing_files)} files not found")
    for f in missing_files:
        print(f"  {f}")
else:
    print(f" All {total_found} slab files found!")

In [ ]:
print("Testing MACE on MgZn₂ (001) surface...")
print("="*70)

# Load MgZn2 (001) slab
test_file = slab_files['MgZn2']['001']
print(f"\nLoading: {test_file}")

test_slab = read_lammps_data(test_file, style='atomic', sort_by_id=True)

print(f"\nStructure info:")
print(f"  Formula: {test_slab.get_chemical_formula()}")
print(f"  Atoms: {len(test_slab)}")
print(f"  Cell: {test_slab.cell.lengths()}")

z_coords = test_slab.positions[:, 2]
print(f"  Z range: {z_coords.min():.2f} - {z_coords.max():.2f} Å")
print(f"  Vacuum: {test_slab.cell[2, 2] - (z_coords.max() - z_coords.min()):.2f} Å")

# Attach MACE calculator
test_slab.calc = mace_calculator

# Single-point energy (no relaxation yet)
print(f"\nCalculating energy with MACE (OMAT/PBE head)...")
energy = test_slab.get_potential_energy()
forces = test_slab.get_forces()

print(f"\n✓ Single-point calculation successful!")
print(f"  Total energy: {energy:.6f} eV")
print(f"  Energy/atom: {energy / len(test_slab):.6f} eV/atom")
print(f"  Max force: {np.max(np.linalg.norm(forces, axis=1)):.6f} eV/Å")
print(f"  Mean force: {np.mean(np.linalg.norm(forces, axis=1)):.6f} eV/Å")

print("\n" + "="*70)
print(" MACE working on real intermetallic surfaces!")

In [ ]:
from ase.constraints import FixAtoms
from ase.optimize import BFGS

def relax_slab_with_mace(slab, calculator, fix_bottom_layers=3.0, fmax=0.01, max_steps=200):
    
    slab_copy = slab.copy()
    slab_copy.calc = calculator
    
    # Fix bottom layers
    z_coords = slab_copy.positions[:, 2]
    z_min = z_coords.min()
    fixed_indices = np.where(z_coords < z_min + fix_bottom_layers)[0]
    
    if len(fixed_indices) > 0:
        constraint = FixAtoms(indices=fixed_indices)
        slab_copy.set_constraint(constraint)
    
    # Initial energy
    E_initial = slab_copy.get_potential_energy()
    
    # Optimize
    optimizer = BFGS(slab_copy, logfile=None)  # No log file
    optimizer.run(fmax=fmax, steps=max_steps)
    
    # Final energy
    E_final = slab_copy.get_potential_energy()
    
    info = {
        'converged': optimizer.converged(),
        'n_steps': optimizer.nsteps,
        'E_initial': E_initial,
        'E_final': E_final,
        'E_change': E_final - E_initial,
        'n_fixed': len(fixed_indices),
        'n_relaxed': len(slab_copy) - len(fixed_indices)
    }
    
    return slab_copy, E_final, info

print("Relaxation function defined")

In [ ]:
# Load MACE Calculator
from mace.calculators import mace_mp
from pathlib import Path

model_path = Path('mace-mh-1.model')

if not model_path.exists():
    print(f"MACE model not found at {model_path}")
else:
    mace_calculator = mace_mp(
        model=str(model_path),
        default_dtype="float64",
        device='cpu',
        head="omat_pbe"
    )
    print(f" MACE loaded from {model_path}")

In [ ]:
# Cell: Calculate Formation Energies - CORRECTED
import json
import pandas as pd

# Load all MACE results
with open('mace_simple_validation_results/mace_simple_validation_results.json') as f:
    solids = json.load(f)

with open('mace_simple_validation_results/molecular_references.json') as f:
    molecules = json.load(f)

print("="*70)
print("FORMATION ENERGIES (MACE vs Experimental)")
print("="*70)

# Extract energies
E_Mg = solids['Mg']['E_final_per_atom']  # per Mg atom
E_Zn = solids['Zn']['E_final_per_atom']  # per Zn atom

E_O2_molecule = molecules['O2']['E_total']  # Total energy of O₂ molecule
E_H2_molecule = molecules['H2']['E_total']  # Total energy of H₂ molecule

print(f"\nReference Energies:")
print(f"  E(Mg) = {E_Mg:.6f} eV/atom")
print(f"  E(Zn) = {E_Zn:.6f} eV/atom")
print(f"  E(O₂) = {E_O2_molecule:.6f} eV/molecule")
print(f"  E(H₂) = {E_H2_molecule:.6f} eV/molecule")

# MgO: Mg + ½O₂ → MgO
E_MgO_solid = solids['MgO']['E_per_formula']
DH_MgO_mace = E_MgO_solid - E_Mg - 0.5*E_O2_molecule
DH_MgO_exp = -6.26  # eV/f.u.

# ZnO: Zn + ½O₂ → ZnO
E_ZnO_solid = solids['ZnO']['E_per_formula']
DH_ZnO_mace = E_ZnO_solid - E_Zn - 0.5*E_O2_molecule
DH_ZnO_exp = -3.61  # eV/f.u.

# Mg(OH)₂: Mg + O₂ + H₂ → Mg(OH)₂
E_MgOH2_solid = solids['Mg(OH)2']['E_per_formula']
DH_MgOH2_mace = E_MgOH2_solid - E_Mg - E_O2_molecule - E_H2_molecule
DH_MgOH2_exp = -9.24  # eV/f.u. (Materials Project)

# MgH₂: Mg + H₂ → MgH₂
E_MgH2_solid = solids['MgH2']['E_per_formula']
DH_MgH2_mace = E_MgH2_solid - E_Mg - E_H2_molecule
DH_MgH2_exp = -0.78  # eV/f.u.

print(f"\nMgO:  {E_MgO_solid:.3f} - {E_Mg:.3f} - 0.5×{E_O2_molecule:.3f} = {DH_MgO_mace:.3f} eV")
print(f"ZnO:  {E_ZnO_solid:.3f} - {E_Zn:.3f} - 0.5×{E_O2_molecule:.3f} = {DH_ZnO_mace:.3f} eV")
print(f"Mg(OH)₂: {E_MgOH2_solid:.3f} - {E_Mg:.3f} - {E_O2_molecule:.3f} - {E_H2_molecule:.3f} = {DH_MgOH2_mace:.3f} eV")
print(f"MgH₂: {E_MgH2_solid:.3f} - {E_Mg:.3f} - {E_H2_molecule:.3f} = {DH_MgH2_mace:.3f} eV")

results = []
for name, mace, exp in [
    ('MgO', DH_MgO_mace, DH_MgO_exp),
    ('ZnO', DH_ZnO_mace, DH_ZnO_exp),
    ('Mg(OH)₂', DH_MgOH2_mace, DH_MgOH2_exp),
    ('MgH₂', DH_MgH2_mace, DH_MgH2_exp)
]:
    error = mace - exp
    pct_error = (error / exp) * 100
    results.append({
        'Compound': name,
        'MACE ΔHf (eV)': round(mace, 3),
        'Exp ΔHf (eV)': exp,
        'Error (eV)': round(error, 3),
        'Error (%)': round(pct_error, 1)
    })

df = pd.DataFrame(results)
print("FORMATION ENERGIES (per formula unit):")
print(df.to_string(index=False))

# Statistics
print("STATISTICS:")
mae = df['Error (eV)'].abs().mean()
mean_pct = df['Error (%)'].abs().mean()
print(f"Mean Absolute Error: {mae:.3f} eV/f.u.")
print(f"Mean % Error: {mean_pct:.1f}%")